In [7]:
import gridstatus
import pandas as pd

In [9]:
caiso = gridstatus.CAISO()

In [155]:
date = pd.to_datetime("2022-10-12 00:00:00")
locations = ["DLAP_SDGE-APND"]

rt_df = caiso.get_lmp(date = date, market = "REAL_TIME_5_MIN", locations = locations, sleep = 5)
hourly_df = caiso.get_lmp(date = date, market = "DAY_AHEAD_HOURLY", locations = locations, sleep = 5)

2026-01-19 20:10:21 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'PRC_INTVL_LMP', 'version': 3}, 'params': {'market_run_id': 'RTM', 'node': None, 'grp_type': [None, 'ALL', 'ALL_APNODES']}}
2026-01-19 20:10:21 - INFO - Fetching URL: http://oasis.caiso.com/oasisapi/SingleZip?resultformat=6&queryname=PRC_INTVL_LMP&version=3&market_run_id=RTM&node=DLAP_SDGE-APND&startdatetime=20221012T07:00-0000&enddatetime=20221013T07:00-0000
2026-01-19 20:10:22 - DEBUG - Found 1 files: ['20221012_20221013_PRC_INTVL_LMP_RTM_20260119_20_10_22_v3.csv']
2026-01-19 20:10:22 - DEBUG - Parsing file: 20221012_20221013_PRC_INTVL_LMP_RTM_20260119_20_10_22_v3.csv
2026-01-19 20:10:27 - DEBUG - Dataset config: {'query': {'path': 'SingleZip', 'resultformat': 6, 'queryname': 'PRC_LMP', 'version': 12}, 'params': {'market_run_id': 'DAM', 'node': None, 'grp_type': [None, 'ALL', 'ALL_APNODES']}}
2026-01-19 20:10:27 - INFO - Fetching URL: http://oasis.caiso.com/oasisapi/SingleZip

In [183]:
rt_df_fixed = rt_df[["Time", "LMP"]].copy()
hourly_df_fixed = hourly_df[["Time", "LMP"]].copy()

In [185]:
rt_df_fixed["Year"] = rt_df_fixed["Time"].dt.year
rt_df_fixed["Month"] = rt_df_fixed["Time"].dt.month
rt_df_fixed["Day"] = rt_df_fixed["Time"].dt.day
rt_df_fixed["Hour"] = rt_df_fixed["Time"].dt.hour

rt_df_fixed = rt_df_fixed[["Month", "Day", "Hour", "LMP", "Year"]]

rt_df_fixed = rt_df_fixed.groupby(["Month", "Day", "Year", "Hour"], as_index=False).agg({"LMP": "mean"})

rt_df_fixed.rename(columns={"LMP": "RT5M_LMP"}, inplace=True)

In [187]:
hourly_df_fixed["Year"] = hourly_df_fixed["Time"].dt.year
hourly_df_fixed["Month"] = hourly_df_fixed["Time"].dt.month
hourly_df_fixed["Day"] = hourly_df_fixed["Time"].dt.day
hourly_df_fixed["Hour"] = hourly_df_fixed["Time"].dt.hour

hourly_df_fixed = hourly_df_fixed[["Month", "Day", "Hour", "LMP", "Year"]]

hourly_df_fixed.rename(columns={"LMP": "DAM_LMP"}, inplace=True)

In [189]:
merged_lmps = rt_df_fixed.merge(hourly_df_fixed, on= ["Year", "Month", "Day", "Hour"])

In [193]:
merged_lmps["RT5M-DAM"] = merged_lmps["RT5M_LMP"] - merged_lmps["DAM_LMP"]

In [195]:
merged_lmps

,Month,Day,Year,Hour,RT5M_LMP,DAM_LMP,RT5M-DAM
0,10,12,2022,0,65.223515,71.10430,-5.880785
1,10,12,2022,1,63.019754,68.11263,-5.092876
2,10,12,2022,2,63.118109,66.91266,-3.794551
3,10,12,2022,3,63.521435,66.96994,-3.448505
4,10,12,2022,4,66.549274,67.19150,-0.642226
5,10,12,2022,5,69.815758,78.19385,-8.378092
6,10,12,2022,6,71.932236,95.40257,-23.470334
7,10,12,2022,7,91.015072,91.25000,-0.234928
8,10,12,2022,8,81.945302,65.59000,16.355302
9,10,12,2022,9,91.801483,70.00000,21.801483


In [201]:
caiso.list_oasis_datasets()

Dataset: transmission_interface_usage
+---------------+-----------+-------------------+
| Parameter     | Default   | Possible Values   |
+===============+===========+===================+
| market_run_id | DAM       | DAM, HASP, RRPD   |
+---------------+-----------+-------------------+
| ti_id         | ALL       | N/A               |
+---------------+-----------+-------------------+
| ti_direction  | ALL       | ALL, E, I         |
+---------------+-----------+-------------------+


Dataset: schedule_by_tie
+-------------+------------------------+-------------------------------------------------------------------------------------------------+
| Parameter   | Default                | Possible Values                                                                                 |
+=============+========================+=================================================================================================+
| groupid     | RTD_ENE_SCH_BY_TIE_GRP | RTD_ENE_SCH_BY_TIE_GRP, DAM